In [1]:
import sys
import os
import json
import tensorflow as tf
import numpy as np
from sngan.generator_gumbel import GumbelGenerator

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

2026-01-28 22:35:57.954190: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-28 22:35:57.981583: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:10575] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-28 22:35:57.981625: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:479] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-28 22:35:57.982921: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1442] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-28 22:35:57.987225: I tensorflow/core/platform/cpu_feature_gua

Num GPUs Available:  2


In [2]:
class FakeFlags:
    model_type = 'wgan'
    architecture = 'gumbel'
    batch_size = 64
    z_dim = 128
    gf_dim = 64
    df_dim = 64
    
    # Kernel / Dilation / Attention
    kernel_height = 3
    kernel_width = 3
    dilation_rate = 2
    attn_pos = 2
    
    # Sequence Config
    seq_len = 160
    vocab_size = 21   
    
    # Misc
    dataset = 'zz'
    logdir = '/project/animesh_ray_1465/Zihao/GAN/logs'
    
    # These values don't affect inference, but are needed to init the class
    generator_learning_rate = 1e-4
    discriminator_learning_rate = 5e-5
    beta1 = 0.5
    beta2 = 0.9
    multid_schedule = 20000
    d_step = 3
    fm_weight = 10
    lambda_gp = 10

    def __getattr__(self, name):
        # Fallback for any flag accessed by code that I didn't explicitly define
        return None

# FLAGS = FakeFlags()
# print(f"Config loaded. Seq Len: {FLAGS.seq_len}")

In [3]:
import os
import json
import tensorflow as tf
import numpy as np
from sngan.generator_gumbel import GumbelGenerator

# ==========================================
# 1. HELPER: Dynamic Flag Loader
# ==========================================
class FlagConfig:
    """Creates an object that acts like FLAGS from a dictionary."""
    def __init__(self, config_dict):
        # Set all json keys as attributes (e.g. flags.hidden_dim)
        for k, v in config_dict.items():
            setattr(self, k, v)
            
    def __getattr__(self, name):
        # Fail-safe: If the code asks for a flag not in the JSON (e.g. older versions)
        # return None instead of crashing.
        return None

def load_flags_from_run(run_dir):
    """
    Tries to find flags.json in 'summaries' (preferred) or root run_dir.
    """
    # 1. Check user-specified location (summaries folder)
    path_primary = os.path.join(run_dir, "summaries", "flags.json")
    
    # 2. Check root location (fallback)
    path_fallback = os.path.join(run_dir, "flags.json")
    
    if os.path.exists(path_primary):
        target_path = path_primary
    elif os.path.exists(path_fallback):
        target_path = path_fallback
    else:
        raise FileNotFoundError(f"Could not find flags.json in {run_dir} or {run_dir}/summaries")
        
    print(f"Loading configuration from: {target_path}")
    with open(target_path, 'r') as f:
        config_dict = json.load(f)
        
    return FlagConfig(config_dict)

# ==========================================
# 2. INFERENCE LOGIC
# ==========================================

def load_and_generate(run_dir, 
                      ckpt_source="best",  # Options: "best" or "latest"
                      pick_rank=1,         # 1 = Top/Newest, 2 = Runner-up/Previous
                      num_samples=50, 
                      temperature=None,    # <--- NEW ARGUMENT
                      output_fasta="generated.fasta"):
    
    # A. Load Flags
    flags = load_flags_from_run(run_dir)
    seq_len = 160
    # B. Initialize
    print(f"Initializing Generator (Seq Len: {seq_len}, Z Dim: {flags.z_dim})...")
    dummy_shape = [1, 1, seq_len, flags.vocab_size]
    g_model = GumbelGenerator(flags, dummy_shape)
    
    # C. Force Build
    print("Building model graph...")
    dummy_z = tf.random.normal([1, flags.z_dim])
    # Build with default just to initialize weights
    _ = g_model(dummy_z, training=False)
    
    # D. Determine Source Directory
    if ckpt_source == "best":
        ckpt_dir = os.path.join(run_dir, "checkpoints", "best_model")
        desc = "BEST FID model"
    else:
        ckpt_dir = os.path.join(run_dir, "checkpoints")
        desc = "LATEST training step"

    # E. Smart Checkpoint Selection
    ckpt = tf.train.Checkpoint(generator=g_model)
    manager = tf.train.CheckpointManager(ckpt, ckpt_dir, max_to_keep=5)
    
    available_ckpts = manager.checkpoints 
    
    if not available_ckpts:
        print(f"CRITICAL ERROR: No checkpoints found in {ckpt_dir}")
        return None
        
    if pick_rank > len(available_ckpts):
        print(f"WARNING: You asked for Rank {pick_rank}, but only {len(available_ckpts)} models exist.")
        target_ckpt = available_ckpts[0]
    else:
        target_ckpt = available_ckpts[-pick_rank]
    
    print(f"--- SELECTION ---")
    print(f"Source: {desc}")
    print(f"Rank:   {pick_rank} (1 is best/newest)")
    print(f"File:   {target_ckpt}")
    print(f"Temp:   {temperature if temperature else 'Default (Fixed 0.5)'}") # print setting
    print(f"-----------------")

    # Restore
    status = ckpt.restore(target_ckpt)
    status.expect_partial() 
    print(f"SUCCESS: Weights restored.")

    # F. Generate (With Uniqueness)
    print(f"Generating {num_samples} UNIQUE sequences...")
    vocab = "ACDEFGHIKLMNPQRSTVWY"
    unique_pool = set()
    attempts = 0
    max_attempts = num_samples * 20
    batch_size = min(64, num_samples)
    
    while len(unique_pool) < num_samples and attempts < max_attempts:
        z = tf.random.normal([batch_size, flags.z_dim])
        
        # --- KEY CHANGE: PASS MANUAL TEMP ---
        if temperature is not None:
             # Use the manual_temp kwarg we added to your Generator class
             probs = g_model(z, training=False, manual_temp=temperature)
        else:
             # Fallback to default behavior
             probs = g_model(z, training=False)
        # ------------------------------------

        indices = tf.argmax(probs, axis=-1).numpy()
        
        for i in range(batch_size):
            flat_idx = indices[i].flatten()
            valid_chars = [vocab[idx-1] for idx in flat_idx if idx != 0]
            seq_str = "".join(valid_chars)
            if len(seq_str) > 10: unique_pool.add(seq_str)
        attempts += batch_size
        print(f"  Collected {len(unique_pool)}/{num_samples}...", end="\r")

    final_seqs = list(unique_pool)[:num_samples]
    
    # G. Save
    if output_fasta:
        with open(output_fasta, "w") as f:
            for i, s in enumerate(final_seqs):
                f.write(f">generated_{i}\n{s}\n")
        print(f"\nSaved to {output_fasta}")
        
    return final_seqs

In [5]:
# ==========================================
# HOW TO USE IT
# ==========================================

RUN_DIR = '/project/animesh_ray_1465/Zihao/GAN/logs/trial4/20251203-153206'

# Scenario 1: Get the Absolute Best (Champion)
seqs = load_and_generate(RUN_DIR, ckpt_source="best", pick_rank=1, num_samples=7000, temperature=0.5, output_fasta=RUN_DIR + "/candidates_best_7k.fasta")

exp_seqs = load_and_generate(RUN_DIR, ckpt_source="best", pick_rank=1, num_samples=7000, temperature=1.0, output_fasta=RUN_DIR + "/gen_high_temp_7k.fasta")

# Scenario 2: Get the Runner-Up (2nd Best FID)
# seqs = load_and_generate(RUN_DIR, ckpt_source="best", pick_rank=2, output_fasta=RUN_DIR + "/candidates_silver.fasta")

# Scenario 3: Get the very last training step (e.g. 40k)
# seqs = load_and_generate(RUN_DIR, ckpt_source="latest", pick_rank=1, output_fasta="candidates_latest.fasta")

Loading configuration from: /project/animesh_ray_1465/Zihao/GAN/logs/trial4/20251203-153206/summaries/flags.json
Initializing Generator (Seq Len: 160, Z Dim: 128)...
Building model graph...
--- SELECTION ---
Source: BEST FID model
Rank:   1 (1 is best/newest)
File:   /project/animesh_ray_1465/Zihao/GAN/logs/trial4/20251203-153206/checkpoints/best_model/ckpt-9999
Temp:   0.5
-----------------
SUCCESS: Weights restored.
Generating 7000 UNIQUE sequences...
  Collected 7040/7000...
Saved to /project/animesh_ray_1465/Zihao/GAN/logs/trial4/20251203-153206/candidates_best_7k.fasta
Loading configuration from: /project/animesh_ray_1465/Zihao/GAN/logs/trial4/20251203-153206/summaries/flags.json
Initializing Generator (Seq Len: 160, Z Dim: 128)...
Building model graph...
--- SELECTION ---
Source: BEST FID model
Rank:   1 (1 is best/newest)
File:   /project/animesh_ray_1465/Zihao/GAN/logs/trial4/20251203-153206/checkpoints/best_model/ckpt-9999
Temp:   1.0
-----------------
SUCCESS: Weights restore